In [1]:
import sys
import os

import torch
import torch.nn as nn

sys.path.append(os.getcwd())
import source
from source import propagator

#%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import time

# Тестирование оптических умножителей матриц

В этом ноутбуке тестируются:
- **OpticalMul** — оригинальная 4f система (POMMM)
- **LumaiMul** — архитектура в стиле Lumai (fan-out + дисплей весов + суммирующая линза)
- **LumaiMulBlocked** — блочное умножение для больших матриц

Метрика качества: **CKO** (среднеквадратичное отклонение, нормированное) — аналог NRMSE:
$$CKO = \sqrt{\left(\frac{C_{optical}}{\overline{C_{optical}}} - \frac{C_{ref}}{\overline{C_{ref}}}\right)^2} \times 100\%$$

## Конфигурация

In [2]:
# ── Параметры для OpticalMul (4f / POMMM) ────────────────────────────────────
right_matrix_count_columns: int = 16
right_matrix_count_rows: int = 16
pixel_size: float = 3.6e-6

config = source.Config(
    right_matrix_count_columns = right_matrix_count_columns,
    right_matrix_count_rows    = right_matrix_count_rows,
    right_matrix_width         = pixel_size * right_matrix_count_columns,
    right_matrix_height        = pixel_size * right_matrix_count_rows,
    min_height_gap             = pixel_size,
    right_matrix_split_x       = 2,
    right_matrix_split_y       = 2,
    left_matrix_split_x        = 2,
    left_matrix_split_y        = 2,
    result_matrix_split        = 2,
    distance                   = 0.01
)

# ── Параметры для LumaiMul ────────────────────────────────────────────────────
# Используем тот же размер матриц (16×16) для честного сравнения
lumai_config = source.LumaiOpticConfig(
    n_lasers         = right_matrix_count_rows,    # M = 16 лазеров
    n_outputs        = right_matrix_count_columns, # N = 16 детекторов
    laser_pitch      = 20e-6,   # 20 мкм — типичный шаг VCSEL матрицы
    detector_pitch   = 20e-6,   # 20 мкм — шаг детектора
    display_pitch    = 8e-6,    # 8 мкм — шаг пикселя дисплея (как Upolabs)
    fanout_distance  = 0.01,    # 1 см — расстояние лазер→дисплей
    summing_distance = 0.01,    # 1 см — расстояние дисплей→детектор
    wavelength       = 850e-9,  # 850 нм — типичная длина волны VCSEL
    display_bits     = 8,       # 8-bit дисплей
    incoherent       = True     # некогерентный свет
)

print(f"Размер матриц: {right_matrix_count_rows}×{right_matrix_count_columns}")
print(f"OpticalMul config: distance={config.distance}, pixel_size={pixel_size}")
print(f"LumaiMul config: n_lasers={lumai_config.n_lasers}, n_outputs={lumai_config.n_outputs}")

Размер матриц: 16×16
OpticalMul config: distance=0.01, pixel_size=3.6e-06
LumaiMul config: n_lasers=16, n_outputs=16


In [7]:
import source, torch; 
c = source.LumaiOpticConfig(n_lasers=16, n_outputs=16); 
m = source.LumaiMul(c); 
A = torch.rand(1,1,8,16); 
B = torch.rand(1,1,16,16); 
print(m(A,B).shape)

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [8, 1] but got: [8, 16].

In [3]:
import source, torch; 
c = source.LumaiOpticConfig(n_lasers=16, n_outputs=16); 
m = source.LumaiMul(c); 
A = torch.rand(1,1,8,16); 
B = torch.rand(1,1,16,16); 
print(m(A,B).shape)

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [8, 1] but got: [8, 16].

## Инициализация моделей

In [10]:
# Оригинальная 4f система
optical_mul = source.OpticalMul(config)
optical_mul.eval()

# Lumai-архитектура
lumai_mul = source.LumaiMul(lumai_config)
lumai_mul.eval()

print("Модели инициализированы")
print(f"  OpticalMul — параметров: {sum(p.numel() for p in optical_mul.parameters())}")
print(f"  LumaiMul   — параметров: {sum(p.numel() for p in lumai_mul.parameters())}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x16 and 1x16)

In [ ]:
# DataParallel (только если доступна CUDA)
if torch.cuda.is_available():
    distributed_mul = source.DataParallel(optical_mul, output_device='cpu')
    print("DataParallel на GPU включён")
else:
    distributed_mul = optical_mul
    print("CUDA недоступна, используем CPU")

## Тест 1: Вещественные матрицы

Сравниваем все три метода на случайных вещественных матрицах.
> Левая матрица — вход слоя, правая — обновляемые веса (сценарий машинного обучения)

In [ ]:
torch.manual_seed(42)
M, K, N = right_matrix_count_rows, right_matrix_count_columns, right_matrix_count_columns
B, C = 1, 1  # batch=1, channels=1

# Случайные вещественные матрицы в формате (B, C, H, W)
A_real = torch.rand((B, C, M, K))
B_real = torch.rand((B, C, K, N))

# Эталон — обычное матричное умножение
with torch.no_grad():
    ref_real = torch.squeeze(A_real) @ torch.squeeze(B_real)

print(f"Входные матрицы: A{list(A_real.shape)}, B{list(B_real.shape)}")
print(f"Эталонный результат: {list(ref_real.shape)}")

In [ ]:
# OpticalMul — вещественные матрицы
with torch.no_grad():
    t0 = time.time()
    res_optical_real = optical_mul(A_real, B_real)
    t_optical = time.time() - t0

res_optical_real = res_optical_real.squeeze()
CKO_optical_real = (((res_optical_real / res_optical_real.mean() - ref_real / ref_real.mean())**2).mean())**0.5 * 100
print(f"OpticalMul (вещественные): CKO = {CKO_optical_real:.4f}%  время = {t_optical*1000:.1f}мс")

In [ ]:
# LumaiMul — вещественные матрицы
with torch.no_grad():
    t0 = time.time()
    res_lumai_real = lumai_mul(A_real, B_real)
    t_lumai = time.time() - t0

res_lumai_real = res_lumai_real.squeeze()
CKO_lumai_real = (((res_lumai_real / res_lumai_real.mean() - ref_real / ref_real.mean())**2).mean())**0.5 * 100
print(f"LumaiMul    (вещественные): CKO = {CKO_lumai_real:.4f}%  время = {t_lumai*1000:.1f}мс")

In [ ]:
# Визуализация — вещественные матрицы
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Вещественные матрицы: результаты умножения')

im0 = axes[0].imshow(ref_real.numpy(), cmap='viridis')
axes[0].set_title('Эталон (torch.matmul)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(res_optical_real.numpy(), cmap='viridis')
axes[1].set_title(f'OpticalMul\nCKO={CKO_optical_real:.2f}%')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(res_lumai_real.numpy(), cmap='viridis')
axes[2].set_title(f'LumaiMul\nCKO={CKO_lumai_real:.2f}%')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## Тест 2: Комплексные матрицы

Тестируем работу с комплексными матрицами (актуально для радарных и коммуникационных приложений).

> **Примечание по LumaiMul**: Lumai использует некогерентный свет, поэтому работает только с амплитудами (`abs()`). Для комплексных матриц это означает потерю фазовой информации — это фундаментальное ограничение архитектуры, а не ошибка реализации.

In [ ]:
torch.manual_seed(42)

# Случайные комплексные матрицы
A_complex = torch.rand((B, C, M, K)) + 1j * torch.rand((B, C, M, K))
B_complex = torch.rand((B, C, K, N)) + 1j * torch.rand((B, C, K, N))

# Эталон
with torch.no_grad():
    ref_complex = torch.squeeze(A_complex) @ torch.squeeze(B_complex)

print(f"Комплексные матрицы: A{list(A_complex.shape)}, B{list(B_complex.shape)}")

In [ ]:
# OpticalMul — комплексные матрицы
with torch.no_grad():
    res_optical_complex = optical_mul(A_complex, B_complex)

res_optical_complex = res_optical_complex.squeeze()
# Сравниваем амплитуды (OpticalMul возвращает abs)
ref_amp = ref_complex.abs()
CKO_optical_complex = (((res_optical_complex / res_optical_complex.mean() - ref_amp / ref_amp.mean())**2).mean())**0.5 * 100
print(f"OpticalMul (комплексные): CKO = {CKO_optical_complex:.4f}%")

In [ ]:
# LumaiMul — комплексные матрицы
# LumaiMul принимает комплексный ввод но работает только с abs() (некогерентная физика)
with torch.no_grad():
    res_lumai_complex = lumai_mul(A_complex, B_complex)

res_lumai_complex = res_lumai_complex.squeeze()
ref_amp_abs = ref_complex.abs()
CKO_lumai_complex = (((res_lumai_complex / res_lumai_complex.mean() - ref_amp_abs / ref_amp_abs.mean())**2).mean())**0.5 * 100
print(f"LumaiMul    (комплексные): CKO = {CKO_lumai_complex:.4f}%")
print("  (LumaiMul сравнивается с |A@B| — фаза теряется из-за некогерентной физики)")

In [ ]:
# Визуализация — комплексные матрицы (амплитуды)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Комплексные матрицы: амплитуда результата')

im0 = axes[0].imshow(ref_complex.abs().numpy(), cmap='viridis')
axes[0].set_title('Эталон |A@B|')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(res_optical_complex.numpy(), cmap='viridis')
axes[1].set_title(f'OpticalMul\nCKO={CKO_optical_complex:.2f}%')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(res_lumai_complex.numpy(), cmap='viridis')
axes[2].set_title(f'LumaiMul (некогер.)\nCKO={CKO_lumai_complex:.2f}%')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

## Тест 3: Масштабирование — зависимость CKO от размера матриц

Ключевой вопрос: при каком размере матрицы точность начинает деградировать?

In [ ]:
sizes = [8, 16, 32, 48, 64]
results_optical = []
results_lumai   = []

for N_size in sizes:
    torch.manual_seed(42)
    A_s = torch.rand(1, 1, N_size, N_size)
    B_s = torch.rand(1, 1, N_size, N_size)
    ref_s = torch.squeeze(A_s) @ torch.squeeze(B_s)

    # OpticalMul
    cfg_s = source.Config(
        right_matrix_count_columns = N_size,
        right_matrix_count_rows    = N_size,
        right_matrix_width         = pixel_size * N_size,
        right_matrix_height        = pixel_size * N_size,
        min_height_gap             = pixel_size,
        right_matrix_split_x       = 2,
        right_matrix_split_y       = 2,
        left_matrix_split_x        = 2,
        left_matrix_split_y        = 2,
        result_matrix_split        = 2,
        distance                   = 0.01
    )
    om = source.OpticalMul(cfg_s)
    om.eval()
    with torch.no_grad():
        res_s = om(A_s, B_s).squeeze()
    cko_s = (((res_s / res_s.mean() - ref_s / ref_s.mean())**2).mean())**0.5 * 100
    results_optical.append(cko_s.item())

    # LumaiMul
    lcfg_s = source.LumaiOpticConfig(
        n_lasers=N_size, n_outputs=N_size,
        laser_pitch=20e-6, detector_pitch=20e-6, display_pitch=8e-6,
        fanout_distance=0.01, summing_distance=0.01,
        wavelength=850e-9, display_bits=8, incoherent=True
    )
    lm = source.LumaiMul(lcfg_s)
    lm.eval()
    with torch.no_grad():
        res_l = lm(A_s, B_s).squeeze()
    cko_l = (((res_l / res_l.mean() - ref_s / ref_s.mean())**2).mean())**0.5 * 100
    results_lumai.append(cko_l.item())

    print(f"N={N_size:<4} | OpticalMul CKO={cko_s:.2f}%  | LumaiMul CKO={cko_l:.2f}%")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sizes, results_optical, 'o-', label='OpticalMul (4f/POMMM)', color='steelblue')
plt.plot(sizes, results_lumai,   's-', label='LumaiMul',              color='coral')
plt.axhline(y=10, color='gray', linestyle='--', alpha=0.7, label='Порог 10% (приемлемо)')
plt.xlabel('Размер матрицы N×N')
plt.ylabel('CKO (%)')
plt.title('Деградация точности с ростом размера матриц')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Тест 4: LumaiMulBlocked — блочное умножение больших матриц

Демонстрирует подход Flash Attention: большие матрицы разбиваются на блоки размером `block_size` и обрабатываются последовательно.

Это ключевая идея для работы с матрицами трансформеров (512×512 и выше).

In [ ]:
# Базовый конфиг блока
BLOCK = 16
lcfg_block = source.LumaiOpticConfig(
    n_lasers=BLOCK, n_outputs=BLOCK,
    laser_pitch=20e-6, detector_pitch=20e-6, display_pitch=8e-6,
    fanout_distance=0.01, summing_distance=0.01,
    wavelength=850e-9, display_bits=8, incoherent=True
)

lumai_blocked = source.LumaiMulBlocked(lcfg_block, block_size=BLOCK)
lumai_blocked.eval()

# Тестируем на матрицах разных размеров
test_sizes = [16, 32, 48, 64]

print(f"Размер блока: {BLOCK}×{BLOCK}")
print()

for N_size in test_sizes:
    torch.manual_seed(42)
    A_b = torch.rand(1, 1, N_size, N_size)
    B_b = torch.rand(1, 1, N_size, N_size)
    ref_b = torch.squeeze(A_b) @ torch.squeeze(B_b)

    with torch.no_grad():
        t0 = time.time()
        res_b = lumai_blocked(A_b, B_b).squeeze()
        t_b = time.time() - t0

    n_shots = (N_size // BLOCK) ** 3
    cko_b = (((res_b / res_b.mean() - ref_b / ref_b.mean())**2).mean())**0.5 * 100

    print(f"N={N_size:<4} | CKO={cko_b:.2f}%  время={t_b*1000:.1f}мс  "
          f"выстрелов={n_shots}  "
          f"при 60Hz={n_shots/60*1000:.0f}мс  "
          f"при 1400Hz={n_shots/1400*1000:.1f}мс")

In [ ]:
# Визуализация результата блочного умножения (последний размер из цикла)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle(f'LumaiMulBlocked: N={N_size}, block={BLOCK}')

im0 = axes[0].imshow(ref_b.numpy(), cmap='viridis')
axes[0].set_title('Эталон (torch.matmul)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(res_b.numpy(), cmap='viridis')
axes[1].set_title(f'LumaiMulBlocked\nCKO={cko_b:.2f}%')
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

## Тест 5: Сравнение когерентного и некогерентного режимов LumaiMul

Проверяем разницу между `incoherent=True` (физически точно для Lumai) и `incoherent=False` (для сравнения с 4f системой).

In [ ]:
torch.manual_seed(42)
A_c = torch.rand(1, 1, 16, 16)
B_c = torch.rand(1, 1, 16, 16)
ref_c = torch.squeeze(A_c) @ torch.squeeze(B_c)

for incoherent, name in [(True, 'Некогерентный (Lumai)'), (False, 'Когерентный (для сравнения)')]:
    cfg_c = source.LumaiOpticConfig(
        n_lasers=16, n_outputs=16,
        laser_pitch=20e-6, detector_pitch=20e-6, display_pitch=8e-6,
        fanout_distance=0.01, summing_distance=0.01,
        wavelength=850e-9, display_bits=8,
        incoherent=incoherent
    )
    lm_c = source.LumaiMul(cfg_c)
    lm_c.eval()
    with torch.no_grad():
        res_c = lm_c(A_c, B_c).squeeze()
    cko_c = (((res_c / res_c.mean() - ref_c / ref_c.mean())**2).mean())**0.5 * 100
    print(f"{name}: CKO = {cko_c:.4f}%")

## Тест 6: Влияние битности дисплея на точность

Моделируем квантование дисплея весов — реальное физическое ограничение установки.

In [ ]:
torch.manual_seed(42)
A_q = torch.rand(1, 1, 16, 16)
B_q = torch.rand(1, 1, 16, 16)
ref_q = torch.squeeze(A_q) @ torch.squeeze(B_q)

bits_list = [4, 6, 8, 10, 12, None]  # None = идеальный дисплей
cko_by_bits = []

for bits in bits_list:
    cfg_q = source.LumaiOpticConfig(
        n_lasers=16, n_outputs=16,
        laser_pitch=20e-6, detector_pitch=20e-6, display_pitch=8e-6,
        fanout_distance=0.01, summing_distance=0.01,
        wavelength=850e-9,
        display_bits=bits if bits is not None else 16,
        incoherent=True
    )
    lm_q = source.LumaiMul(cfg_q)
    lm_q.eval()
    with torch.no_grad():
        res_q = lm_q(A_q, B_q).squeeze()
    cko_q = (((res_q / res_q.mean() - ref_q / ref_q.mean())**2).mean())**0.5 * 100
    cko_by_bits.append(cko_q.item())
    label = f"{bits}-bit" if bits else "идеал"
    print(f"  {label:<12}: CKO = {cko_q:.4f}%")

In [ ]:
plt.figure(figsize=(8, 4))
x_labels = [f"{b}-bit" if b else "ideal" for b in bits_list]
plt.bar(x_labels, cko_by_bits, color='steelblue', alpha=0.8)
plt.axhline(y=10, color='red', linestyle='--', alpha=0.7, label='10% порог')
plt.xlabel('Битность дисплея весов')
plt.ylabel('CKO (%)')
plt.title('Влияние квантования дисплея на точность LumaiMul')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Итоговое сравнение

| Модель | Вещественные | Комплексные | Физика | Масштаб |
|--------|-------------|-------------|--------|---------|
| **OpticalMul (4f/POMMM)** | ✅ | ✅ | 4f, когерентный, SLM | ~50×50 реально |
| **LumaiMul** | ✅ | ⚠️ (только \|A\|) | fan-out+дисплей, некогерентный | 1024×2048 заявлено |
| **LumaiMulBlocked** | ✅ | ⚠️ | блоки + суммирование | любой размер |

**Ключевые выводы:**
- OpticalMul работает с комплексными матрицами — это преимущество 4f архитектуры
- LumaiMul некогерентен — теряет фазовую информацию, но масштабируется на большие матрицы
- Блочное умножение позволяет работать с матрицами любого размера ценой `(N/block)³` оптических тактов